# 🧠 Attention Mechanisms
**Course:** Deep Learning  
**Lecture:** 06 — Attention Mechanisms and Transformers  
**Part:** a — Attention Mechanism

---

> **Where we are:** Sequence-to-sequence models with RNNs struggle with long contexts. Attention allows models to focus on relevant parts of the input dynamically.

**What you'll learn:**
- Additive, Multiplicative, and Scaled Dot-Product Attention
- The mathematical foundations of Attention
- How scaling prevents softmax saturation
- Masking techniques

**Exam relevance:** ⭐⭐⭐⭐⭐ (Understanding attention is critical for all modern NLP questions)


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns

# Confirm device
print(f"TensorFlow {tf.__version__} | Devices: {[d.name for d in tf.config.list_physical_devices()]}")

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12


## 📖 Theory: Three Major Attention Variants

1. **Additive Attention (Bahdanau, 2014)**
   - Formula: $e_{ij} = v^T \tanh(W_1 h_i + W_2 s_j)$
   - Uses a feed-forward network to learn alignment. Flexible but parameter-heavy and slower.

2. **Multiplicative / Dot-Product Attention (Luong, 2015)**
   - Formula: $e_{ij} = h_i^T s_j$
   - Uses simple dot product. Fast and parameter-free, but variance grows with dimension.

3. **Scaled Dot-Product Attention (Vaswani, 2017 — Transformers)**
   - Formula: $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$
   - $Q$: Query (what we are looking for)
   - $K$: Key (what we match against)
   - $V$: Value (what we extract)
   - Divides by $\sqrt{d_k}$ to prevent softmax saturation in high dimensions. This is the foundation of all Transformers.


## 🔀 Your Options

| Option | Pros | Cons | When to use |
|---|---|---|---|
| Additive | Highly expressive | Slower, extra weights | Complex, low-resource tasks |
| Multiplicative | Extremely fast | Unstable with large $d_k$ | Small dimension embeddings |
| Scaled Dot-Product | Fast, stable | Assumes specific alignment | Transformers, standard NLP |

At this step, you can choose between: Additive, Multiplicative, or Scaled Dot-Product. We will implement Scaled Dot-Product as it is the most ubiquitous.


In [ ]:
def numpy_scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]                              # key dimension for scaling
    scores = Q @ K.T / np.sqrt(d_k)               # dot product then scale
    # Softmax: convert scores to probabilities that sum to 1
    exp_scores = np.exp(scores - scores.max(axis=-1, keepdims=True))  # subtract max for stability
    weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)      # normalize
    output = weights @ V                           # weighted sum of values
    return output, weights

output, weights = numpy_scaled_dot_product_attention(Q_dummy, K_dummy, V_dummy)
print(f'Input  Q shape: {Q_dummy.shape}  K: {K_dummy.shape}  V: {V_dummy.shape}')
print(f'Output shape:   {output.shape}')
print(f'Attention weights shape: {weights.shape}')
print(f'Weights sum per row: {weights.sum(axis=-1).round(4)}  (must all be 1.0)')

# Visualise attention as heatmap
plt.figure(figsize=(5, 4))
plt.imshow(weights, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(label='Attention weight')
plt.xlabel('Key position'); plt.ylabel('Query position')
plt.title('Attention weight matrix')
plt.tight_layout(); plt.show()

In [ ]:
def numpy_scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]                              # key dimension for scaling
    scores = Q @ K.T / np.sqrt(d_k)               # dot product then scale
    # Softmax: convert scores to probabilities that sum to 1
    exp_scores = np.exp(scores - scores.max(axis=-1, keepdims=True))  # subtract max for stability
    weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)      # normalize
    output = weights @ V                           # weighted sum of values
    return output, weights

output, weights = numpy_scaled_dot_product_attention(Q_dummy, K_dummy, V_dummy)
print(f'Input  Q shape: {Q_dummy.shape}  K: {K_dummy.shape}  V: {V_dummy.shape}')
print(f'Output shape:   {output.shape}')
print(f'Attention weights shape: {weights.shape}')
print(f'Weights sum per row: {weights.sum(axis=-1).round(4)}  (must all be 1.0)')

# Visualise attention as heatmap
plt.figure(figsize=(5, 4))
plt.imshow(weights, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(label='Attention weight')
plt.xlabel('Key position'); plt.ylabel('Query position')
plt.title('Attention weight matrix')
plt.tight_layout(); plt.show()

In [ ]:
def tf_scaled_dot_product_attention(q, k, v, mask=None):
    d_k = tf.cast(tf.shape(k)[-1], tf.float32)
    scores = tf.matmul(q, k, transpose_b=True) / tf.sqrt(d_k)  # (batch, seq, seq)
    if mask is not None:
        # Add large negative value to masked positions so softmax gives ~0
        scores = scores + (mask * -1e9)
    weights = tf.nn.softmax(scores, axis=-1)  # probabilities along key axis
    output = tf.matmul(weights, v)
    return output, weights

# Test without mask
q_tf = tf.constant(Q_dummy[np.newaxis], dtype=tf.float32)  # add batch dim
k_tf = tf.constant(K_dummy[np.newaxis], dtype=tf.float32)
v_tf = tf.constant(V_dummy[np.newaxis], dtype=tf.float32)
out_tf, w_tf = tf_scaled_dot_product_attention(q_tf, k_tf, v_tf)
print(f'TF attention output shape: {out_tf.shape}')

# Test causal mask (each position can only attend to previous positions)
seq_len = 3
causal_mask = 1 - tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)  # upper triangle = 1
print(f'Causal mask:\n{causal_mask.numpy()}')
_, w_masked = tf_scaled_dot_product_attention(q_tf, k_tf, v_tf, mask=causal_mask)
print(f'Weights with causal mask (upper triangle should be ~0):\n{w_masked.numpy()[0].round(3)}')

In [ ]:
def tf_scaled_dot_product_attention(q, k, v, mask=None):
    d_k = tf.cast(tf.shape(k)[-1], tf.float32)
    scores = tf.matmul(q, k, transpose_b=True) / tf.sqrt(d_k)  # (batch, seq, seq)
    if mask is not None:
        # Add large negative value to masked positions so softmax gives ~0
        scores = scores + (mask * -1e9)
    weights = tf.nn.softmax(scores, axis=-1)  # probabilities along key axis
    output = tf.matmul(weights, v)
    return output, weights

# Test without mask
q_tf = tf.constant(Q_dummy[np.newaxis], dtype=tf.float32)  # add batch dim
k_tf = tf.constant(K_dummy[np.newaxis], dtype=tf.float32)
v_tf = tf.constant(V_dummy[np.newaxis], dtype=tf.float32)
out_tf, w_tf = tf_scaled_dot_product_attention(q_tf, k_tf, v_tf)
print(f'TF attention output shape: {out_tf.shape}')

# Test causal mask (each position can only attend to previous positions)
seq_len = 3
causal_mask = 1 - tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)  # upper triangle = 1
print(f'Causal mask:\n{causal_mask.numpy()}')
_, w_masked = tf_scaled_dot_product_attention(q_tf, k_tf, v_tf, mask=causal_mask)
print(f'Weights with causal mask (upper triangle should be ~0):\n{w_masked.numpy()[0].round(3)}')

In [ ]:
# 🔬 Comparison: Heatmaps and Scaling effects

q = tf.random.normal((1, 5, 64))
k = tf.random.normal((1, 5, 64))
v = tf.random.normal((1, 5, 64))

# 1. Without scaling (Multiplicative)
logits_unscaled = tf.matmul(q, k, transpose_b=True)
weights_unscaled = tf.nn.softmax(logits_unscaled, axis=-1)

# 2. With scaling
_, weights_scaled = tf_scaled_dot_product_attention(q, k, v)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(weights_unscaled[0], ax=axes[0], cmap='viridis', vmin=0, vmax=1)
axes[0].set_title("Without Scaling (Saturated)")
sns.heatmap(weights_scaled[0] if len(weights_scaled.shape)==3 else weights_scaled, ax=axes[1], cmap='viridis', vmin=0, vmax=1)
axes[1].set_title("With sqrt(d_k) Scaling")
plt.show()


## 🎓 Exam Corner

### Question 1 (Conceptual)
Write the scaled dot-product attention formula and explain what Q, K, V represent. Why do we divide by $\sqrt{d_k}$? What happens if we don't?

<details><summary>💡 Answer</summary>

**Formula:** $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$

- **Q (Query):** The current element we are processing, looking for relevant info.
- **K (Key):** The elements we match against to compute relevance scores.
- **V (Value):** The actual content we extract once we have the scores.

We divide by $\sqrt{d_k}$ to prevent the dot products from growing too large. If we don't, the softmax function will be pushed into regions where it has extremely small gradients (saturation), making learning very slow.

</details>

### Question 2 (Practical)
What is the shape of the attention weight matrix given Q: (seq_len, d_k) and K: (seq_len, d_k)? What is the difference between self-attention and cross-attention?

<details><summary>💡 Answer</summary>

**Shape:** `(seq_len, seq_len)` - It's a square matrix where each row represents a query token and each column represents a key token.

**Difference:**
- **Self-attention:** Q, K, and V all come from the exact same sequence.
- **Cross-attention:** Q comes from the target/decoder sequence, while K and V come from the source/encoder sequence.

</details>
